<a href="https://colab.research.google.com/github/nullpointer3202/CU_OnlineJudge_Result/blob/main/oj_selenium_headless_git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 셀레니움 및 한글 폰트 설치
!apt-get update
!apt install -y fonts-nanum
!fc-cache -fv
!rm ~/.cache/matplotlib -rf

# Google Chrome 설치
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -O /tmp/google-chrome-stable_current_amd64.deb
!dpkg -i /tmp/google-chrome-stable_current_amd64.deb
!apt-get install -f -

# ChromeDriver 설치
# Google Chrome 버전 <=> ChromeDriver 버전
!CHROMIUM_VERSION=$(/usr/bin/google-chrome --version 2>/dev/null | cut -d ' ' -f 3 | cut -d '.' -f 1)

!if [ -z "$CHROMIUM_VERSION" ]; then \
    echo "/usr/bin/google-chrome에서 버전 확인 불가능. (122버전으로 시도)"; \
    CHROMIUM_VERSION="122"; \
fi

!wget -N https://edgedl.me.gvt1.com/edgedl/chrome/chrome-for-testing/$(wget -qO- https://googlechromelabs.github.io/chrome-for-testing/last-known-good-versions-with-downloads.json | grep -oP "v${CHROMIUM_VERSION}.*?" | grep -oP "[0-9.]+" | head -1)/linux64/chromedriver-linux64.zip -P /tmp/
!unzip -o /tmp/chromedriver-linux64.zip -d /tmp/
!mv /tmp/chromedriver-linux64/chromedriver /usr/bin/chromedriver
!chmod +x /usr/bin/chromedriver

!pip install selenium

In [ ]:
##########################################################################
##                                                                      ##
##      P L T   한 글 폰 트   설 정 용   런 타 임   재 시 작   구 간    ##
##                                                                      ##
##########################################################################

In [ ]:
import selenium
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

import numpy as np

from selenium import webdriver
from selenium.webdriver import ActionChains

from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By

from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait

from selenium.webdriver.common.alert import Alert

from selenium.webdriver.chrome.service import Service

#한글폰트 지정
font_path = "/usr/share/fonts/truetype/nanum/NanumGothicCoding.ttf"  # 현재 시스템에 있는 경로로 변경
font_name = fm.FontProperties(fname=font_path).get_name()

plt.rc('font', family='NanumGothicCoding')
plt.rcParams['axes.unicode_minus'] =False

In [ ]:
font_name


'NanumGothicCoding'

In [ ]:
# OJ 사이트에 로그인하는 코드
def oj_login(driver):
    id_box = driver.find_element(By.XPATH, '//*[@id="id"]')
    pwd_box = driver.find_element(By.XPATH, '//*[@id="password"]')
    login_btn = driver.find_element(
        By.XPATH, "/html/body/div[2]/div/div/div/div/form/div[2]/input"
    )
    id_box.send_keys(input("자신의 아이디"))
    pwd_box.send_keys(input("자신의 비밀번호"))

    login_btn.click()

    try:
        print(driver.get_issue_message)

        if driver.get_issue_message:
            print(Alert(driver).text)
            driver.quit()
    except:
        print("PASS")







# 과목 선택
def get_course(driver):
    #과목 리스트 찾기
    myclass_element = driver.find_element(By.CLASS_NAME, "list-group.list-group-bar")

    #각 과목들을 찾는 코드
    courses = myclass_element.find_elements(By.CLASS_NAME, "list-group-item")


    for idx, course in enumerate(courses):
        print(f"[{idx+1}] {course.text}")

    # 데이터를 가져올 과목 선택
    selected_course_idx = int(input("데이터를 가져올 과목 : "))
    selected_course = courses[selected_course_idx-1]
    course_name = selected_course.text
    # 과목 클릭
    selected_course.click()
    return course_name

#과제(실습 or 시험) 선택
def get_assignments(driver):
    assignments = driver.find_elements(
        By.XPATH, "/html/body/div[2]/div/div[1]/div/ul/li[position()>1]"
    )
    for idx, assignment in enumerate(assignments):
        print(f"[{idx+1}] {assignment.text}")
    selected_assignment_idx = int(input("데이터를 가져올 문제유형 : "))
    selected_assignment = assignments[selected_assignment_idx - 1]
    assignment_name = ''.join(selected_assignment.text)
    selected_assignment.click()
    print("선택한 문제유형 : ", assignment_name)
    return assignment_name




# 제출정보 모으기 => get_data에서 사용예정!
def get_submissions(driver, data):
    submissions = driver.find_elements(
        By.XPATH, '//*[@id="result-tab"]/tbody/tr[position()>0]'
    )

    for idx, submission in enumerate(submissions):
        sub_suid = submission.find_element(By.XPATH, ".//td[2]").text

        try:
            sub_result = submission.find_element(By.XPATH, ".//td[4]/btn").text
        except:
            sub_result = submission.find_element(By.XPATH, ".//td[4]").text
        sub_score = submission.find_element(By.XPATH, ".//td[5]").text


        #가져오고 있는 데이터를 출력
        #print(sub_suid, sub_result, sub_score)
        if (
            sub_result == "Empty Test-data"
            or sub_result == "Judging"
            or sub_result == "Compile Error"
        ):
            continue
        write_data(sub_suid, sub_result, float(sub_score.replace(" /100", "")), data)




#각 문제의 제출정보 열기
def get_status(problem, driver):
    problem_name = problem.find_element(By.XPATH, ".//td[2]/a")
    problem_status = problem.find_element(By.XPATH, ".//td[7]/a")
    problem_status.click()

    uid_box = driver.find_element(By.XPATH, '//*[@id="uid"]')
    search_btn = driver.find_element(
        By.XPATH, "/html/body/div[2]/div/div[4]/div[1]/form/input"
    )

    # 아이디 박스 내용 삭제
    uid_box.clear()
    search_btn.click()


#제출 정보 내의 모든 데이터를 모으기
def get_data(driver):
    #print("DEBUG : get_data 실행")
    data = []

    try : # 테스트 결과 페이지 수 확인 후 LINK 생성
        pagelist = driver.find_element(
            By.XPATH, "/html/body/div[2]/div/div[4]/div[3]/div/nav/ul"
        )
    except : # 테스트 횟수가 일정 수를 넘지 않아 페이지 구분이 생기지 않으면 해당 페이지만 실행
        get_submissions(driver, data)
        return data

    pager = pagelist.find_elements(By.XPATH, ".//li[position()>0]")

    # 제출 상태의 마지막 페이지 찾기
    lastpage = pager[-1]
    last_page_num = lastpage.find_element(By.XPATH, ".//a").get_attribute(
        "data-ci-pagination-page"
    )

    #현재 데이터를 수집하고있는 페이지 표시
    print("page ", end= '')


    # 페이지가 끝날 때까지 제출 상태를 출력
    for page in range(1, int(last_page_num) + 1):
        print(f"{page} ", end = '')
        get_submissions(driver, data)  # 데이터 불러오는 함수
        pagelist = driver.find_element(
            By.XPATH, "/html/body/div[2]/div/div[4]/div[3]/div/nav/ul"
        )
        pager = pagelist.find_elements(By.XPATH, ".//li[position()>0]")

        #페이지가 끝나면 break
        try:
            next = driver.find_element(
                By.XPATH, f".//a[@data-ci-pagination-page='{page+1}']"
            )

            nextpage = next.get_attribute("href")
            #print("페이지 이동!!!! ", nextpage)
            driver.get(nextpage)
        except:
            break
    print()
    return data

#Data 작성 및 수정 : 같은 아이디가 있을 경우 대체. => get_submissions에서 사용
def write_data(suid, result, score, data):
    for student in data:
        if student["suid"] == suid:
            if student["score"] < score:
                student["result"] = result
                student["score"] = score
            return
    data.append({"suid": suid, "result": result, "score": score})




#datalist에 저장된 데이터 출력
def print_data(datalist):
    for code_index, data_list in enumerate(datalist):
        print(f"문제 {code_index + 1}에 대한 점수:")
        for i, data in enumerate(data_list):
            print(f"Data {i + 1}:")
            print(f"아이디: {data['suid']}")
            print(f"상태: {data['result']}")
            print(f"점수: {data['score']}")
            print()

#학생당 문제별 시도 여부 출력
def print_trial(datalist):
    student_data = {}
    student_trial = {}
    for code_index, data_list in enumerate(datalist):
        for data in data_list:
            student_num = str(data["suid"])
            status = data["result"]
            score = data["score"]

            if student_num in student_data:
                # 기존 학생 번호에 대한 값 업데이트
                student_data[student_num]["count"] += 1
                student_data[student_num]["total_score"] += score

                # 상태가 "Accept"인 경우, 맞은 갯수 업데이트
                if status == "Accept":
                    student_data[student_num]["accept_count"] += 1
            else:
                # student data(딕셔너리 타입)에 학생 번호 추가
                student_data[student_num] = {
                    "count": 1,
                    "accept_count": 1 if status == "Accept" else 0,
                    "total_score": score,
                }

            #시도한 문제들
            if student_num in student_trial:
                student_trial[student_num].append(code_index)
            else:
                student_trial[student_num] = [code_index]

    student_data = dict(sorted(student_data.items(), key=lambda x: x[1]["total_score"], reverse=True))



    #print("AFTER", student_data)
    calc = 0
    for student_num, data in student_data.items():
        calc += 1
        print(calc)
        print(f"아이디: {student_num}")
        print(f"맞은 갯수: {data['accept_count']}")
        print(f"총점: {data['total_score']}")

        # 학생이 시도한 문제 출력
        print("시도한 문제들:")
        for code_index in student_trial[student_num]:
            print(f"  문제 {code_index+1}")

        print()

#학생당 문제별 시도 여부 저장
def get_trial_data(student_data, student_trial, datalist):
    for code_index, data_list in enumerate(datalist):
        for data in data_list:
            student_num = str(data["suid"])
            status = data["result"]
            score = data["score"]

            if student_num in student_data:
                # 기존 학생 번호에 대한 값 업데이트
                student_data[student_num]["count"] += 1
                student_data[student_num]["total_score"] += score

                # 상태가 "Accept"인 경우, 개수 업데이트
                if status == "Accept":
                    student_data[student_num]["accept_count"] += 1
            else:
                # 딕셔너리에 학생 번호를 위한 새로운 항목 추가
                student_data[student_num] = {
                    "count": 1,
                    "accept_count": 1 if status == "Accept" else 0,
                    "total_score": score,
                }

            if student_num in student_trial:
                student_trial[student_num].append(code_index)
            else:
                student_trial[student_num] = [code_index]

    tdata = []
    for student_num, data in student_data.items():
        student = {}
        student["아이디"] = student_num
        student["맞은 갯수"] = data["accept_count"]
        student["총점"] = data["total_score"]

        student["시도한 문제들"] = []
        for code_index in student_trial[student_num]:
            student["시도한 문제들"].append(f"문제 {code_index+1}")

        tdata.append(student)
    return tdata

#
def totalscores(trialdata):
    score_counts = {}
    for data in trialdata:
        score = data["총점"]
        if score in score_counts:
            score_counts[score] += 1
        else:
            score_counts[score] = 1
    return score_counts


################################################################################
#
#                 그래프 표현 함수 공간
#
################################################################################


def students_each_prob_graph(datalist, cname, aname):
    problem_attempt_count = {}
    problem_correct_count = {}

    # 각 HTML 코드의 데이터 리스트를 반복
    for code_index, data_list in enumerate(datalist):
        # 코드 블록별로 딕셔너리 초기화
        problem_attempt_count[code_index] = {}
        problem_correct_count[code_index] = {}

        # 데이터 리스트 내의 각 데이터를 반복
        for data in data_list:
            student_num = data["suid"]

            # 문제를 시도한 학생 수 업데이트
            if student_num in problem_attempt_count[code_index]:
                problem_attempt_count[code_index][student_num] += 1
                if data['result'] == "Accept":
                    problem_correct_count[code_index][student_num] = 1
            else:
                problem_attempt_count[code_index][student_num] = 1
                if data['result'] == "Accept":
                    problem_correct_count[code_index][student_num] = 1

    # 문제 시도 횟수를 오름차순으로 정렬한 딕셔너리 생성
    sorted_problem_counts = sorted(problem_attempt_count.items(), key=lambda x: x[0])
    sorted_correct_counts = sorted(problem_correct_count.items(), key=lambda x: x[0])

    # 문제 인덱스와 학생 수를 따로 리스트로 추출
    problem_indices = [problem_index for problem_index, _ in sorted_problem_counts]
    student_counts = [sum(counts.values()) for _, counts in sorted_problem_counts]

    c_problem_indices = [problem_index for problem_index, _ in sorted_correct_counts]
    correct_counts = [sum(counts.values()) for _, counts in sorted_correct_counts]

    # 각 문제에 대한 학생 수를 나타내는 막대 그래프 생성
    bar = plt.bar(problem_indices, student_counts)
    correct_bar = plt.bar(c_problem_indices, correct_counts, label="정답자 수", alpha=0.7)

    for rect in bar:
        height = rect.get_height()
        plt.text(rect.get_x() + rect.get_width()/2.0, height, '%d' % height, ha='center', va='bottom', size=12)

    for rect in correct_bar:
        height = rect.get_height()
        plt.text(rect.get_x() + rect.get_width()/2.0, height, '%d' % height, ha='center', va='bottom', size=12)

    plt.legend()
    plt.xlabel("문제 번호")
    plt.ylabel("학생 수")
    plt.title(f"{cname} {aname}의 각 문제를 시도한 학생 수")
    plt.show()




def score_distribution_graph(trialdata, cname, aname):
    score_counts = totalscores(trialdata)

    plt.figure(figsize=(10, 5))
    bar = plt.bar(score_counts.keys(), score_counts.values())

    for rect in bar:
        height = rect.get_height()
        plt.text(rect.get_x() + rect.get_width()/2.0, height, '%d' % height, ha='center', va='bottom', size = 12)

    plt.xlabel("점수")
    plt.ylabel("학생 수")
    plt.title(f"{cname} {aname}의 점수 분포표")

    plt.show()


def total_score_graph(trialdata, cname, aname):

    graphthick = 30  # 그래프의 두께와 세부도. 10~50, 30이 적절
    toppercent = 20  # 그래프의 상위 퍼센트 하이라이트. 25%가 적절
    titleText = f"{cname} {aname} 응시자의 총 점수 분포 (상위 {toppercent}%)"  # 그래프 제목, 한글불가

    total_scores = []
    for data in trialdata:
        total_scores.append(data["총점"])

    # 총 학생 수
    total_students = len(total_scores)

    # 총점 분포도 그래프 그리기
    plt.hist(total_scores, bins=graphthick, edgecolor="black")
    plt.yticks(np.arange(0, 10))
    plt.xlabel("총점")
    plt.ylabel("학생 수")
    plt.title(f"{titleText}")
    plt.text(
        0.05,
        0.95,
        f"학생 수: {total_students}",
        transform=plt.gca().transAxes,
    )

    total_scores_np = np.array(total_scores)
    threshold = np.percentile(total_scores_np, 100 - toppercent)
    plt.axvspan(threshold, np.max(total_scores), color="orange", alpha=0.3)

    plt.show()

In [ ]:
from selenium import webdriver
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

In [ ]:
URL = "https://oj.chosun.ac.kr/index.php/auth/login/"

#크롬 드라이버 초기화 및 실행
driver = webdriver.Chrome(options=chrome_options)

#URL 열기
driver.get(url = URL)

oj_login(driver)


############################# 과목 선택 페이지로 이동 ########################


#과목 선택 및 이름 가져오기
course_name = get_course(driver)


############################# 해당 과목 페이지로 이동 ########################

#과제 선택 및 이름 가져오기
assignment_name = get_assignments(driver)


############################# 해당 과제 페이지로 이동 ########################

#돌아오기용 페이지 위치
problems_page = driver.current_url

#과제 내 문제들을 찾는 코드
problems = driver.find_elements(By.XPATH,'/html/body/div[2]/div/div[4]/div/table/tbody/tr[position()>0]')

datalist = []

for idx, problem in enumerate(problems) :

    prob = driver.find_elements(By.XPATH,'/html/body/div[2]/div/div[4]/div/table/tbody/tr[position()>0]')


    #문제 이름들을 찾아내 출력하는 코드
    problem_name = prob[idx].find_element(By.XPATH, './/td[2]/a')
    problem_status = prob[idx].find_element(By.XPATH, './/td[7]/a')
    print(f"\n[{idx+1}] {problem_name.text}")

    print("데이터 받아오는 중....")
    try :
        get_status(prob[idx], driver)
        datalist.append(get_data(driver))
    except Exception as e :
        print(e)
    #print(datalist[idx])

    driver.get(problems_page)

datalistdev = datalist
print(datalist)


In [ ]:
#학생별 맞은 갯수와 총점, 시도한 문제
student_data = {}
student_trial = {}

print_trial(datalist)

trialdata = get_trial_data(student_data, student_trial, datalist)

In [ ]:
datalist

In [ ]:
plt.rc('font', family='NanumGothicCoding')

students_each_prob_graph(datalist,course_name, assignment_name)

score_distribution_graph(trialdata, course_name, assignment_name)

total_score_graph(trialdata, course_name, assignment_name)



#한글 폰트가 적용이 안된다면 런타임 재시작